# Regulatory Sandbox — Dev Log

## Objetivo e papel no pipeline

`core/regulatory_sandbox` responde perguntas "e se...?" compondo
`policy_engine` + `trust_score` reais, **sem** gravar nada em `audit_logs` —
a diferença estrutural em relação ao `ripd_engine` (que sempre grava). Útil
para explorar cenários hipotéticos sem gerar ruído na trilha de auditoria de
produção.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.regulatory_sandbox.sandbox import compare_scenarios
from shared.schemas import DataCategory, LegalBasis, SandboxScenario

sem_revisao = SandboxScenario(
    name="Biometria sem revisão humana",
    data_categories=[DataCategory.SENSITIVE],
    legal_basis=LegalBasis.NOT_DETERMINED,
    context={"data_subtype": "biometric", "automated_decision": True, "human_review": False},
)
com_revisao = SandboxScenario(
    name="Biometria COM revisão humana",
    data_categories=[DataCategory.SENSITIVE],
    legal_basis=LegalBasis.NOT_DETERMINED,
    context={"data_subtype": "biometric", "automated_decision": True, "human_review": True},
)
comparison = compare_scenarios(sem_revisao, com_revisao)
print(f"Score A ({comparison.scenario_a.scenario_name}): {comparison.scenario_a.trust_score.score}")
print(f"Score B ({comparison.scenario_b.scenario_name}): {comparison.scenario_b.trust_score.score}")
print(f"Delta: {comparison.score_delta}")
print("Decisões que deixam de se aplicar:", comparison.decisions_removed)
print("Novas decisões:", comparison.decisions_added)
print()
print(comparison.summary)

Score A (Biometria sem revisão humana): 5.0
Score B (Biometria COM revisão humana): 70.0
Delta: 65.0
Decisões que deixam de se aplicar: []
Novas decisões: []

Cenário 'Biometria sem revisão humana' (5.0) -> 'Biometria COM revisão humana' (70.0): melhora o trust score em 65.0 ponto(s).


Habilitar `human_review=True` sobe o trust score de 5.0 (piso do `DENY`
real) para 70.0 — uma resposta concreta e mensurável para "o que eu preciso
mudar para este projeto deixar de ser bloqueado?", sem precisar rodar o
`ripd_engine` completo (e sem sujar a trilha de auditoria com simulações).

## Rodando a suíte de testes

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/regulatory_sandbox/tests -v
```

7 testes: cenário de baixo risco; cenário de alto risco com `DENY` real;
**prova explícita de ausência de efeito colateral** (contagem de eventos de
`audit_logs` antes/depois de `simulate()` idêntica); delta de score correto;
decisões adicionadas/removidas; resumo menciona os nomes dos cenários;
cenários idênticos com delta zero.

## Handoff Summary

- **Status:** ✅ done — 7/7 testes passando.
- **Consumível por:** um futuro botão "simular cenário" no
  `apps/dashboard`, ou por `governance_copilot` como endpoint
  `POST /api/v1/sandbox/compare` (não implementado nesta onda, TODO).